# Lab 7: Naive Bayes — From Scratch to Bayes Optimal
### (Instructor Copy — Reference Solutions)

**Objective:** Classify sentences from biomedical abstracts into their rhetorical role
(BACKGROUND, OBJECTIVE, METHODS, RESULTS, CONCLUSIONS). Students implement Multinomial Naive
Bayes in NumPy, verify it against scikit-learn, then explore smoothing, feature representation
and model combination.

**Dataset:** PubMed 20k RCT — 180,040 labelled training sentences.

---

**Instructor copy.** Every `TODO` appears with a worked solution and every reflection prompt
with a model answer. Figures quoted are from `PES1UG23CS999`; only Part C depends on the SRN
(it sets the training subsample size), so Parts A and B should reproduce for every student.
Expected total runtime is about 2 minutes.

---
## 1. Setup & Your Assignment

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix)

plt.rcParams["figure.dpi"] = 110

Set `STUDENT_ID` to **your own SRN**. Its last three digits set the size of the training
subsample used in Part C, so your Part C numbers will differ from your classmates'.

In [ ]:
STUDENT_ID = "PES1UG23CS999"   # <-- CHANGE THIS TO YOUR SRN

last_three = int(STUDENT_ID[-3:])
SAMPLE_SIZE = 10_000 + last_three
SEED = 42

print(f"Student ID       : {STUDENT_ID}")
print(f"Part C subsample : {SAMPLE_SIZE:,} training sentences")

---
## 2. Load the Data

Fully worked — run as-is.

Upload `train.txt`, `dev.txt` and `test.txt` from the PubMed 20k RCT dataset next to this
notebook first. Each line is a label and a sentence separated by a tab; lines starting with
`###` are abstract IDs and are skipped.

The dataset already ships as **three** files, so the split is made for you. Keep the roles
straight, because Section 7 depends on it:

| Split | Purpose |
|---|---|
| **train** | Fits the model — counts the words. |
| **dev** | Chooses hyperparameters (`alpha`, n-gram range). |
| **test** | Scored **once**, at the end. Never used to make a choice. |

In [ ]:
def load_pubmed_rct_file(filepath):
    """Read one PubMed 20k RCT .txt file into a DataFrame of (label, sentence)."""
    labels, sentences = [], []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or "\t" not in line:
                continue                      # skips '###<id>' lines and blanks
            label, sent = line.split("\t", maxsplit=1)
            labels.append(label)
            sentences.append(sent)
    return pd.DataFrame({"label": labels, "sentence": sentences})


dir_path = "./"
train_df = load_pubmed_rct_file(os.path.join(dir_path, "train.txt"))
dev_df   = load_pubmed_rct_file(os.path.join(dir_path, "dev.txt"))
test_df  = load_pubmed_rct_file(os.path.join(dir_path, "test.txt"))

X_train, y_train = train_df["sentence"], train_df["label"]
X_dev,   y_dev   = dev_df["sentence"],   dev_df["label"]
X_test,  y_test  = test_df["sentence"],  test_df["label"]
target_names = sorted(y_train.unique())

print(f"Train : {len(train_df):,}")
print(f"Dev   : {len(dev_df):,}")
print(f"Test  : {len(test_df):,}")
print(f"Classes: {target_names}")

In [ ]:
dist = y_train.value_counts(normalize=True).reindex(target_names)

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(dist.index, dist.values, color="#4c72b0")
ax.set_ylabel("share of training sentences")
ax.set_title("Class distribution")
ax.grid(alpha=0.3, axis="y")
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

print(dist.round(3).to_string())
print(f"\nAlways-predict-'{dist.idxmax()}' accuracy would be {dist.max():.3f}")
print("Two example sentences:")
for lab in target_names[:2]:
    print(f"  [{lab}] {train_df[train_df.label == lab].sentence.iloc[0][:90]}...")

**TODO (reflection):** The classes are far from balanced. Using the number printed above,
explain why plain **accuracy** is a weak way to score this task, and what **macro-averaged F1**
(which averages the per-class F1 scores, weighting every class equally) measures instead.
Which class do you expect to be hardest, and why?

*Your answer (model):* METHODS alone is 33% of the training data, so a model that predicts
METHODS for every sentence and learns nothing at all scores 0.33 accuracy. Accuracy is
dominated by the two large classes (METHODS and RESULTS together are 65% of the data), so a
model can look respectable while failing completely on the small ones. Macro-F1 averages the
per-class F1 scores with equal weight, so OBJECTIVE — at 7.7% of the data — counts as much as
METHODS; a model that ignores it is penalised in a way accuracy hides. OBJECTIVE should be
hardest: it is both the rarest class and semantically close to BACKGROUND, since both appear
in the opening sentences of an abstract and share much of their vocabulary. Expect the two
biggest confusions in Section 5 to be OBJECTIVE/BACKGROUND and METHODS/RESULTS.

---
## 3. Multinomial Naive Bayes, From Scratch

### Concept

Naive Bayes picks the class with the highest posterior probability given the words in the
sentence. By Bayes' rule, and dropping the evidence term (the same for every class):

$$\hat{c} = \arg\max_{c}\; P(c) \prod_{i} P(w_i \mid c)^{x_i}$$

where $x_i$ is how many times word $i$ appears. The "naive" part is that product: it assumes
words are **conditionally independent** given the class, which is plainly false for language
("randomized" and "controlled" are not independent) but works well anyway.

**Everything happens in log space.** A sentence of 25 words multiplies 25 probabilities, each
perhaps $10^{-5}$ — the result underflows to exactly 0.0 in floating point, and every class
ties at zero. Taking logs turns the product into a sum, which is numerically stable and faster:

$$\log P(c) + \sum_i x_i \log P(w_i \mid c)$$

**Laplace (additive) smoothing.** If a word never appears in class $c$ in training, its
probability is 0, the product is 0, and that class is ruled out by a single unseen word — no
matter how strongly the other 24 words support it. Smoothing adds a pseudo-count $\alpha$ to
every word:

$$P(w_i \mid c) = \frac{\text{count}(w_i, c) + \alpha}{\sum_j \text{count}(w_j, c) + \alpha V}$$

where $V$ is the vocabulary size. The $\alpha V$ in the denominator keeps it a valid
probability distribution. Section 6 asks you to tune $\alpha$.

### What to implement

Both methods are **fully vectorised** — no Python loop over sentences. Store the learned
parameters in two arrays, both ordered to match `self.classes_`:

- `self.class_log_prior_` — shape `(n_classes,)`
- `self.feature_log_prob_` — shape `(n_classes, vocab_size)`

Those are the names scikit-learn uses, which is what makes the Section 4 check a one-liner.

In [ ]:
class NaiveBayesClassifier:
    """Multinomial Naive Bayes with Laplace smoothing, implemented from scratch."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    def fit(self, X, y):
        """X: sparse count matrix (n_samples, vocab_size).  y: array-like of labels."""
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        self.vocabulary_size_ = X.shape[1]
        n_samples = X.shape[0]

        log_priors, log_likelihoods = [], []
        for c in self.classes_:
            X_c = X[y == c]                   # rows belonging to this class

            # ---- (1) log prior for class c ---------------------------------
            log_priors.append(np.log(X_c.shape[0] / n_samples))

            # counts[j] = how many times word j occurs across all of class c
            counts = np.asarray(X_c.sum(axis=0)).ravel()

            # ---- (2) smoothed log likelihood for class c -------------------
            numerator = counts + self.alpha
            denominator = counts.sum() + self.alpha * self.vocabulary_size_
            log_likelihoods.append(np.log(numerator / denominator))

        # ---- (3) stack into arrays -----------------------------------------
        self.class_log_prior_ = np.array(log_priors)
        self.feature_log_prob_ = np.vstack(log_likelihoods)

        return self

    def predict_log_scores(self, X):
        """Un-normalised log posterior for every class. Shape (n_samples, n_classes)."""
        # ---- (4) one matrix multiply, no loops -----------------------------
        return X @ self.feature_log_prob_.T + self.class_log_prior_

    def predict(self, X):
        """Return the predicted label for every row of X."""
        # ---- (5) argmax across the class axis, then map index -> label -----
        return self.classes_[np.argmax(self.predict_log_scores(X), axis=1)]


---
## 4. Build the Features & Verify Your Implementation

First turn sentences into count vectors. `CountVectorizer` is fitted on the **training split
only** — fitting it on everything would let the vocabulary be shaped by sentences the model is
not supposed to have seen.

`min_df=2` drops words appearing in only one sentence across 180,000. Those are typos and
one-off technical terms; keeping them inflates the vocabulary (and therefore the $\alpha V$
smoothing term) without adding signal.

In [ ]:
count_vectorizer = CountVectorizer(
    lowercase=True,
    strip_accents="unicode",
    stop_words="english",
    ngram_range=(1, 1),
    min_df=2,
)

X_train_counts = count_vectorizer.fit_transform(X_train)   # fit on TRAIN only
X_dev_counts   = count_vectorizer.transform(X_dev)
X_test_counts  = count_vectorizer.transform(X_test)

print(f"Vocabulary size: {X_train_counts.shape[1]:,}")
print(f"Train matrix   : {X_train_counts.shape}, "
      f"{X_train_counts.nnz / np.prod(X_train_counts.shape):.4%} non-zero")


### The verification check — provided, do not modify

This is the most important cell in Part A.

scikit-learn's `MultinomialNB` computes exactly the same quantities with exactly the same
smoothing, so if your implementation is correct, your two parameter arrays must match its own
to floating-point precision. Run this before you look at a single accuracy score.

The reason to check this way: **a wrong Naive Bayes still classifies.** Forget the $\alpha V$
term in the denominator and your probabilities no longer sum to one — but they are still
monotone in the counts, so the argmax is often unchanged and accuracy barely moves. Drop the
log prior entirely and you lose a couple of points, which looks like a modelling choice rather
than a bug. Neither failure announces itself.

In [ ]:
_mine = NaiveBayesClassifier(alpha=1.0).fit(X_train_counts, y_train)
_skl  = MultinomialNB(alpha=1.0).fit(X_train_counts, y_train)

assert hasattr(_mine, "class_log_prior_"), "fit() never set class_log_prior_"
assert hasattr(_mine, "feature_log_prob_"), "fit() never set feature_log_prob_"
assert _mine.feature_log_prob_.shape == _skl.feature_log_prob_.shape, (
    f"feature_log_prob_ is {_mine.feature_log_prob_.shape}, "
    f"expected {_skl.feature_log_prob_.shape} (n_classes, vocab_size)")

prior_err = np.abs(_mine.class_log_prior_ - _skl.class_log_prior_).max()
lik_err   = np.abs(_mine.feature_log_prob_ - _skl.feature_log_prob_).max()
print(f"Max difference in log priors      : {prior_err:.3e}")
print(f"Max difference in log likelihoods : {lik_err:.3e}")

assert prior_err < 1e-10, (
    "LOG PRIORS DO NOT MATCH. P(c) is the share of training ROWS in class c "
    "(X_c.shape[0] / n_samples) — not the share of word counts.")
assert lik_err < 1e-10, (
    "LOG LIKELIHOODS DO NOT MATCH. Most likely the denominator: it must be "
    "counts.sum() + alpha * vocabulary_size, and the alpha*V term is easy to omit.")

_agree = (_mine.predict(X_test_counts) == _skl.predict(X_test_counts)).mean()
print(f"Prediction agreement on test      : {_agree:.4%}")
assert _agree > 0.999, "Parameters match but predictions do not — check predict()."

print("\nSection 4 checks passed — your implementation is correct.")

**TODO (reflection):** Suppose you had forgotten the `alpha * vocabulary_size` term in the
denominator. The check above would fail, but would the **accuracy** have dropped noticeably?
Reason about what that term does to the scores of each class, and explain why matching a
reference implementation is a stronger test than "the accuracy looks about right".

*Your answer (model):* Barely. Dropping `alpha * V` divides every likelihood in class c by the
same constant, which subtracts a constant from that class's log score — so the per-class
ranking of *sentences* is untouched, and only the comparison *between* classes shifts, by a
constant offset that depends on each class's total word count. The result is a small, uniform
bias toward classes with more total words; accuracy moves by a fraction of a point and the
confusion matrix still looks sensible. The model would be silently wrong: its outputs are no
longer probabilities, they do not sum to one, and the "probabilities" could not be used for
anything downstream — including the posterior weights in Section 8, which would be quietly
miscomputed.

That is the general point. Accuracy is a weak test because it only sees the argmax, and the
argmax survives a great many errors in the quantities behind it. Comparing against a reference
implementation tests the actual numbers, catches errors that leave the decisions intact, and
either passes or fails — it has no judgement call in it. Accuracy "looking about right" only
tells you the bug was not catastrophic.

---
## 5. Baseline Evaluation

Train your classifier on the count features and score it on the test set.

In [ ]:
RESULTS = []    # every experiment appends one row here


def log_result(name, y_true, y_pred, note="", elapsed=None):
    """Score a set of predictions, print a summary, and record it in RESULTS."""
    row = {
        "Experiment": name,
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Macro F1": round(f1_score(y_true, y_pred, average="macro"), 4),
        "Time (s)": None if elapsed is None else round(elapsed, 1),
        "Observations": note,
    }
    RESULTS.append(row)
    print(f"  {name}: accuracy {row['Accuracy']:.4f} | macro-F1 {row['Macro F1']:.4f}")
    return row


def plot_confusion(y_true, y_pred, title):
    """Row-normalised confusion matrix: each row shows where that true class went."""
    cm = confusion_matrix(y_true, y_pred, labels=target_names)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)

    plt.figure(figsize=(6.5, 5))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=target_names, yticklabels=target_names,
                vmin=0, vmax=1, cbar_kws={"label": "share of true class"})
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.title(title)
    plt.tight_layout(); plt.show()


print("Helpers ready.")

In [ ]:
t0 = time.time()
nb_model = NaiveBayesClassifier(alpha=1.0).fit(X_train_counts, y_train)
y_test_pred = nb_model.predict(X_test_counts)
elapsed = time.time() - t0

log_result("Custom NB (counts, alpha=1)", y_test, y_test_pred,
           note="Baseline. OBJECTIVE is the weakest class, as expected.",
           elapsed=elapsed)

print(classification_report(y_test, y_test_pred, target_names=target_names))
plot_confusion(y_test, y_test_pred, "Custom NB — counts, alpha=1")


**TODO (reflection):** Look at the row-normalised confusion matrix. Name the two largest
off-diagonal confusions and explain, in terms of the words those classes share, why Naive
Bayes makes them. Does the pattern match what you predicted in Section 2?

*Your answer (model):* The two dominant confusions are OBJECTIVE being predicted as BACKGROUND,
and RESULTS being predicted as METHODS. Both are vocabulary overlaps that the bag-of-words
assumption cannot resolve. OBJECTIVE and BACKGROUND both open the abstract and share
"study", "aim", "patients", "investigate", "effect"; the distinction is often carried by
sentence *position* and by tense rather than by word choice, and the model sees neither — it
gets an unordered bag of words with no idea where in the abstract the sentence sat. RESULTS and
METHODS share the entire apparatus of trial reporting: group names, measurement names,
"patients", "group", "score", numbers. OBJECTIVE suffers additionally from being the rarest
class, so its log prior is the lowest and ties get broken against it. This matches the Section 2
prediction. The most useful feature the model is missing is positional — which is the natural
next step for this dataset and the motivation for sequential sentence-classification models.

---
# Part B — Smoothing and Features

## 6. Experiment 1 — The Smoothing Parameter

### Concept

$\alpha$ controls how much probability mass is reserved for words a class has never produced.
Small $\alpha$ trusts the training counts almost completely, so a single unseen word can
dominate the score; large $\alpha$ flattens the word distributions toward uniform, washing out
the differences between classes. Somewhere in between is a best value, and the **dev set** is
where you find it — using the test set to pick $\alpha$ would make the final score a
self-fulfilling one.

In [ ]:
alpha_scores = {}

for alpha in [0.01, 0.1, 0.5, 1.0, 2.0]:
    m = NaiveBayesClassifier(alpha=alpha).fit(X_train_counts, y_train)
    alpha_scores[alpha] = f1_score(y_dev, m.predict(X_dev_counts), average="macro")
    print(f"  alpha = {alpha:<5} dev macro-F1 = {alpha_scores[alpha]:.4f}")

best_alpha = max(alpha_scores, key=alpha_scores.get)
print(f"\nBest alpha on dev: {best_alpha}")

plt.figure(figsize=(6.5, 3.8))
plt.plot(list(alpha_scores), list(alpha_scores.values()), "o-", lw=1.8)
plt.axvline(best_alpha, color="crimson", ls="--", lw=1, label=f"best = {best_alpha}")
plt.xscale("log")
plt.xlabel("alpha (log scale)"); plt.ylabel("dev macro-F1")
plt.title("Smoothing strength vs dev performance")
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Only now touch the test set.
tuned = NaiveBayesClassifier(alpha=best_alpha).fit(X_train_counts, y_train)
log_result(f"Custom NB (counts, tuned alpha={best_alpha})", y_test,
           tuned.predict(X_test_counts),
           note=f"Alpha chosen on dev (={best_alpha}), scored once on test.")


**TODO (reflection):** `alpha=0.01` should score clearly worse than `alpha=1.0`, even though
it fits the training counts more faithfully. Explain what goes wrong when smoothing is too
weak. Then explain why the whole sweep had to run on the dev set — what exactly would be wrong
with the final test number if you had picked `alpha` by testing each value on the test set?

*Your answer (model):* With `alpha=0.01` an unseen word gets a likelihood around
0.01/(total + 0.01V) instead of 1/(total + V) — roughly a hundred times smaller, which is a
log penalty of about −4.6 applied to that class. Because the scores are a *sum* of those log
terms, one or two unseen words can outweigh the combined evidence of every other word in the
sentence, so a class is effectively vetoed by vocabulary it merely happens not to have produced
in training. This hurts the rare classes most: OBJECTIVE has the fewest training words, so it
has the most unseen vocabulary and collects the most vetoes — which is why the damage shows up
in macro-F1 more than in accuracy. Larger alpha makes the penalty for an unseen word mild and
lets the rest of the sentence be heard.

On the protocol: nothing about the mechanics would differ — you would get a number, and it
would be slightly *higher*. The problem is what it would mean. Picking the alpha that maximises
the test score makes the test set part of the fitting procedure, so the resulting figure is the
best of five attempts rather than an estimate of performance on unseen data, and it is
optimistically biased by an amount you cannot measure from inside the experiment. With five
values and a large test set the bias here is small, but that is luck, not method: the size of
the bias grows with how many choices you make against the test set, and it is never something
you can report.

---
## 7. Experiment 2 — Counts vs TF-IDF

### Concept

Raw counts treat every occurrence of every word as equally informative. **TF-IDF** rescales
each count by how rare the word is across the corpus, so "patients" — which appears everywhere
and separates nothing — is damped, while a word concentrated in one class keeps its weight.

It is the standard default for text classification, so the interesting question is whether it
actually helps *here*. Note that TF-IDF produces fractional values, not counts, so the
multinomial model is no longer being fed the kind of data its derivation assumes; it still
works, but the justification is now empirical rather than principled.

Use scikit-learn's `MultinomialNB` from here on — your own implementation has been verified
to be identical, and the pipeline machinery is what the rest of this section is about.

In [ ]:
# --- Step 1: TF-IDF with default settings -------------------------------------
t0 = time.time()
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, strip_accents="unicode",
                              stop_words="english", min_df=2)),
    ("nb", MultinomialNB()),
])
pipeline.fit(X_train, y_train)
log_result("Sklearn NB (TF-IDF, default)", y_test, pipeline.predict(X_test),
           note="TF-IDF untuned — worse than counts.", elapsed=time.time() - t0)

# --- Step 2: tune on the dev set ----------------------------------------------
t0 = time.time()
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "nb__alpha": [0.1, 0.5, 1.0],
}
grid = GridSearchCV(pipeline, param_grid, cv=3, scoring="f1_macro", n_jobs=-1)
grid.fit(X_dev, y_dev)

print(f"Best parameters : {grid.best_params_}")
print(f"Best dev CV F1  : {grid.best_score_:.4f}")
print(f"Grid search took {time.time() - t0:.1f}s")

# --- Step 3: refit the winner on the full training set ------------------------
t0 = time.time()
best_model = grid.best_estimator_.fit(X_train, y_train)
log_result("Sklearn NB (TF-IDF, tuned)", y_test, best_model.predict(X_test),
           note="Bigrams + low alpha; best single model in the lab.",
           elapsed=time.time() - t0)

plot_confusion(y_test, best_model.predict(X_test), "Tuned TF-IDF Naive Bayes")


**TODO (reflection):** The **default** TF-IDF model should score *worse* than your plain
count model from Section 6, which is the opposite of what its reputation suggests — and then
tuning should push it past both.

(a) Why does untuned TF-IDF lose to raw counts here? Think about what TF-IDF values look like
compared to counts, and what `alpha=1.0` does to each.

(b) What did adding bigrams buy, and what does it cost?

*Your answer (model):* (a) The culprit is the interaction between TF-IDF's scale and a fixed
alpha. TF-IDF values are L2-normalised fractions — typically well under 1 — whereas counts are
integers of 1 or more. The smoothing term added to each feature is `alpha=1.0` in both cases,
but relative to the data it is enormous for TF-IDF: a pseudo-count of 1 can exceed the entire
real signal for a feature, so the word distributions are flattened toward uniform and the
classes become hard to tell apart. Macro-F1 falls further than accuracy because the rare
classes are the first to be washed out. Note this is not really a verdict on TF-IDF — it is a
verdict on carrying over a hyperparameter from a different feature scale, which is why the
grid search tunes alpha and the representation *together*. Dropping alpha to 0.1 is most of
the repair.

(b) Bigrams let the model see short phrases as single features — "double blind",
"adverse events", "statistically significant", "we conclude" — several of which are strong,
almost deterministic signals for one class in a way their component words are not. The costs
are a vocabulary several times larger (so more memory and a slower fit), many bigrams seen only
a handful of times, and a larger V inflating the `alpha * V` smoothing term. It pays off here
because the training set is large enough to estimate the useful bigrams; on a few thousand
sentences it typically would not.

---
# Part C — The Bayes Optimal Classifier

## 8. Combining Hypotheses by Their Posterior

### Concept

Everything so far picks a single model and uses it. The **Bayes Optimal Classifier** says that
is not the best you can do. Rather than committing to one hypothesis $h$, it averages the
predictions of *every* hypothesis, each weighted by how probable that hypothesis is given the
data:

$$P(c \mid x, D) = \sum_{h \in H} P(c \mid x, h)\, P(h \mid D)$$

No classifier using the same hypothesis space can do better on average — which is what
"optimal" means here. The catch is that $H$ is the space of *all* hypotheses, which is
infinite, so the true BOC is not computable. What we build instead is an approximation:
take a handful of trained models as a stand-in for $H$, estimate each one's posterior weight
$P(h \mid D)$, and combine their predicted probabilities by **soft voting**.

**Estimating $P(h \mid D)$.** With equal priors over the four models, $P(h \mid D)$ is
proportional to the likelihood of held-out data under $h$. So: hold out part of the training
sample, compute each model's average log-likelihood of the true labels on it, and turn those
into weights with a softmax. Section 8 also asks you to compute the weights the *exact* way —
with the summed log-likelihood instead of the mean — and compare.

Part C uses a subsample of `SAMPLE_SIZE` training sentences (from your SRN), so these models
are weaker than the full-data models in Parts A and B. Compare them with each other, not with
earlier rows.

In [ ]:
tfidf_params = dict(lowercase=True, strip_accents="unicode", stop_words="english",
                    ngram_range=(1, 1), min_df=5)

hypotheses = {
    "NaiveBayes": Pipeline([("tfidf", TfidfVectorizer(**tfidf_params)),
                            ("clf", MultinomialNB(alpha=0.5))]),
    "LogisticRegression": Pipeline([("tfidf", TfidfVectorizer(**tfidf_params)),
                                    ("clf", LogisticRegression(max_iter=1000,
                                                               random_state=SEED))]),
    "DecisionTree": Pipeline([("tfidf", TfidfVectorizer(**tfidf_params)),
                              ("clf", DecisionTreeClassifier(max_depth=20,
                                                             random_state=SEED))]),
    "RandomForest": Pipeline([("tfidf", TfidfVectorizer(**tfidf_params)),
                              ("clf", RandomForestClassifier(n_estimators=60, max_depth=25,
                                                             random_state=SEED, n_jobs=-1))]),
}

X_sample = X_train[:SAMPLE_SIZE]
y_sample = y_train[:SAMPLE_SIZE]
print(f"Part C training subsample: {len(X_sample):,} sentences")
print(f"Hypothesis space H: {list(hypotheses)}")

In [ ]:
X_fit, X_held, y_fit, y_held = train_test_split(
    X_sample, y_sample, test_size=0.25, random_state=SEED, stratify=y_sample)

mean_log_lik = {}
for name, model in hypotheses.items():
    model.fit(X_fit, y_fit)
    proba = np.clip(model.predict_proba(X_held), 1e-12, None)
    col = [list(model.classes_).index(label) for label in y_held]
    true_p = proba[np.arange(len(y_held)), col]
    mean_log_lik[name] = np.mean(np.log(true_p))
    print(f"  {name:<20} mean log-lik {mean_log_lik[name]:>8.4f} | "
          f"held-out acc {model.score(X_held, y_held):.4f}")

lls = np.array([mean_log_lik[n] for n in hypotheses])
w = np.exp(lls - lls.max())
posterior_weights = w / w.sum()

print("\nPosterior weights P(h | D), from MEAN log-likelihood:")
for name, weight in zip(hypotheses, posterior_weights):
    print(f"  {name:<20} {weight:.4f}")

# The exact version: the likelihood of the whole held-out set is the SUM of the
# per-row log-likelihoods, not their mean.
total_lls = lls * len(y_held)
w_exact = np.exp(total_lls - total_lls.max())
w_exact = w_exact / w_exact.sum()
print("\nPosterior weights from the SUM (the exact Bayesian quantity):")
for name, weight in zip(hypotheses, w_exact):
    print(f"  {name:<20} {weight:.6f}")


In [ ]:
t0 = time.time()
boc = VotingClassifier(estimators=list(hypotheses.items()), voting="soft",
                       weights=list(posterior_weights), n_jobs=-1)
boc.fit(X_sample, y_sample)
log_result("BOC (weighted soft vote)", y_test, boc.predict(X_test),
           note="Posterior-weighted ensemble of 4 hypotheses.",
           elapsed=time.time() - t0)

print("\nIndividual hypotheses, trained on the same subsample:")
for name, model in hypotheses.items():
    model.fit(X_sample, y_sample)
    log_result(f"  {name} (alone)", y_test, model.predict(X_test),
               note="Part C base model.")


**TODO (reflection):** Two things here, and the second is the important one.

(a) Compare the BOC's macro-F1 against each individual hypothesis. Did the ensemble beat all of
its members? If not, look at the weights — which models received substantial weight, and how
did they perform alone?

(b) The two weight calculations give very different answers: the mean-log-likelihood softmax
spreads weight across the models, while the summed version puts essentially **all** of it on
one. The summed version is the mathematically correct $P(h \mid D)$. Explain why it collapses,
what that collapse implies about the Bayes Optimal Classifier when $H$ contains only a handful
of hypotheses, and which of the two would have scored better here.

*Your answer (model):* (a) No — the BOC (macro-F1 ≈ 0.590) lost to Logistic Regression alone
(≈ 0.625). The reason is visible in the weights: the mean-LL softmax gave LR 0.35 but also gave
Random Forest 0.27 and Naive Bayes 0.33, even though RF alone manages only ≈ 0.409 and NB
≈ 0.549. Those weights are nearly uniform, so the ensemble is close to an unweighted average and
the weak members outvote the strong one. Soft voting only helps when members make *different*
mistakes; here they largely make the same ones, so averaging adds no correction and just dilutes
the best model.

(b) The collapse is arithmetic, not a bug. The log-likelihood of the whole held-out set is the
**sum** over its rows, so with about 2,750 held-out sentences a gap of 0.05 in mean
log-likelihood becomes a gap of roughly 135 nats in total — and the weights are proportional to
`exp` of that, a factor of around e¹³⁵. Any difference at all between two models, multiplied by
thousands of observations, makes one of them astronomically more probable. This is correct
Bayesian behaviour: with enough data the posterior concentrates on the single best hypothesis in
the set.

The implication is that with a small, finite `H`, the Bayes Optimal Classifier degenerates into
"pick the best model" — it only averages meaningfully when many hypotheses are genuinely
near-indistinguishable given the data, which is the situation its theory has in mind. And the
exact weights would have scored *better* here, since they put all the weight on Logistic
Regression (≈ 0.625 vs the ensemble's ≈ 0.590). The mean-log-likelihood softmax is a
*tempered* posterior: dividing by the sample size is a deliberate flattening that keeps the
ensemble alive, and it is a heuristic, not the theory. It is worth being clear that averaging
was the thing that cost us performance here — and that "optimal" in the BOC's name is a
statement about averaging over the *true* posterior across the full hypothesis space, not a
promise that any weighted vote beats its best member.

---
## 9. Results Table

In [ ]:
results_df = pd.DataFrame(RESULTS).sort_values("Macro F1", ascending=False)
results_df = results_df.reset_index(drop=True)
results_df.to_csv("lab7_results.csv", index=False)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)
print(results_df.to_string(index=False))
results_df


---
## 10. Reflection

Answer each in 3-5 sentences, referring to **your own numbers**.

1. **Naive Bayes assumes words are conditionally independent given the class, which is plainly
   false for real language.** Give a concrete example of two words in this dataset that are
   obviously dependent, explain what the model does wrong because of it, and suggest why the
   classifier still works reasonably well despite the assumption being violated.

   *Model answer:* "Randomized" and "controlled" almost always co-occur in this corpus, as do
   "confidence" and "interval", or "double" and "blind". Naive Bayes treats each as independent
   evidence, so a sentence containing "randomized controlled trial" contributes three separate
   log-likelihood terms for METHODS when there is really only one fact — the phrase being
   present. The model therefore becomes **overconfident**: its posterior probabilities are
   pushed much closer to 0 and 1 than the evidence warrants. It still classifies well because
   the argmax only needs the *ranking* of the classes to be right, and over-counting correlated
   evidence usually inflates the score of the class that was already winning. The practical
   consequence is that Naive Bayes is a decent classifier but a poor probability estimator —
   which matters directly in Section 8, where its `predict_proba` output feeds the soft vote.

2. **Rank the three things you varied — smoothing strength, feature representation, and model
   combination — by how much each changed your macro-F1.** Which gave the best return for the
   effort involved?

   *Model answer:* Feature representation moved it most (TF-IDF with bigrams and a tuned alpha
   reached 0.663 against 0.657 for counts — but the *untuned* TF-IDF fell to 0.578, so the swing
   across this axis was by far the largest). Smoothing was second, worth about two points between
   alpha 0.01 and 1.0, and more on the rare classes than the common ones. Model combination
   returned the least: the BOC scored 0.590, below Logistic Regression's 0.625 on the same
   subsample, so it actively cost performance. The best return for effort was tuning alpha — one
   cheap loop over five values, no new concepts, and it is also what rescues TF-IDF from being
   worse than raw counts. Accept any ranking consistent with the student's own table.

3. **The dev set was used to choose `alpha` and the n-gram range; the test set was scored only
   after those choices were fixed.** Explain what would have been wrong with the reported test
   figure if you had used the test set for tuning, and why the problem is not fixed by the test
   set being large.

   *Model answer:* The figure would be the maximum over the settings tried rather than an
   estimate of performance on unseen data, and a maximum over several noisy measurements is
   biased upward. Once the test set has chosen a hyperparameter it has participated in fitting
   the model, so it no longer answers the question a test score is supposed to answer. Size
   helps with *variance* — each individual estimate is more precise with 30,000 sentences — but
   it does nothing about this bias, which comes from the selection step rather than from noise,
   and which grows with the number of choices made. The honest workflow is the one used here:
   every decision is made against dev, and test is read once, after the decisions are frozen.

4. **You verified your implementation against scikit-learn before measuring its accuracy.**
   Describe one bug that check would catch which a reasonable-looking accuracy score would not,
   and explain why that kind of silent error is harder to find in a classifier than a crash.

   *Model answer:* Omitting the `alpha * V` term from the denominator is the clean example: it
   subtracts a per-class constant from every log score, so the within-class ranking is untouched
   and accuracy moves by a fraction of a point, while the outputs stop being probabilities
   altogether. Computing the prior from total word counts rather than row counts is another —
   it shifts the balance toward wordy classes by a small, plausible-looking amount. Both leave a
   model that trains, predicts, and produces a confusion matrix with the shape you would expect.
   A crash tells you exactly where to look; a silent numerical error gives you a working system
   whose only symptom is a score slightly lower than it should have been, which is
   indistinguishable from an ordinary modelling limitation. The reference comparison replaces
   that judgement call with a pass/fail on the numbers themselves.

---

### Marking notes

- **Sections 3-4 are objective**: the verification cell passes or it does not. A student whose
  `prior_err` or `lik_err` exceeds `1e-10` has not completed Part A, whatever their accuracy.
- **Expected figures** (SRN-independent for Parts A and B): custom NB on counts,
  accuracy 0.7226 / macro-F1 0.6567; best alpha on dev is 1.0; untuned TF-IDF 0.6950 / 0.5783;
  tuned TF-IDF (bigrams, alpha 0.1) 0.7416 / 0.6632. Part C varies with SRN, but the ordering
  should hold: LogisticRegression (≈0.625) > BOC (≈0.590) > NaiveBayes (≈0.549) >
  RandomForest (≈0.409) > DecisionTree (≈0.346).
- **The BOC losing to its best member is the expected result**, not a student error. Mark down
  answers that report the ensemble as a success without checking it against the individual rows.
- **Common silent bugs**: forgetting `alpha * V`; computing the prior from word counts rather
  than row counts; returning `np.argmax(...)` directly from `predict()` instead of indexing
  into `self.classes_` (gives integer predictions, which `accuracy_score` scores as 0).
- Runtime is about 2 minutes total; the grid search (~15 s) and Part C (~30 s) dominate.